# Datamine SURFIP Process Example

This notebook demonstrates how to configure and run the **`surfip`** process wrapper in `dmstudio`.

## Process Description

## SURFIP

Interpolates a set of point data into a pair of surfaces expressed as sub-cells in a 3-D model with any number of cells in the Z direction.

The surfaces are modeled as sub-cell divisions in the Z direction, at the exact surface position. **SURFIP** also allows simultaneous division of cells into sub-cells in the X and Y directions, under user control, in places where the local slope of the surface is steep.

The use of an optional trend file permits interpolation of residuals around a trend surface, and faults and sub-crop limits may be specified by use of an optional perimeter file defining interpolation regions.

### Input data and interpolation

Data are entered from file &**IN** as a series of X and Y co-ordinates together with elevations of upper and lower surfaces. These are the * **X** , * **Y** , * **UPPER** and * **LOWER** fields. A prototype model is required to define the model parameters (the &**PROTO** file), but no data records from the prototype model are used (even if they exist), since **SURFIP** constructs an entirely new model.

This prototype file may be constructed by process [PROTOM](<protom.md>), or it may be a previously existing model. Note that **SURFIP** always represents its surfaces as sub-cells, and therefore the **XINC** , **YINC** and **ZINC** fields should be explicit (sub-cells used); however, if the input prototype should contain implicit values of these fields, they are converted to explicit automatically.

The elevation of the upper surface and the distance between the surfaces are interpolated using the inverse power of distance method. The optional @**POWER** parameter controls this power; note that 0 should not be used; however, averaging of points may be effectively obtained by setting @**POWER** to some small value, such as 0.00001. The higher the value of @**POWER** , the more the output surface will be close to the input data points near the points. However, values of @**POWER** greater than about 5 may give rise to arithmetic errors during processing as numbers become too large. The position of the lower surface is calculated by subtraction of the interpolated thickness from the top. This method prevents any cross-overs of the top and bottom surfaces.

The sample data file usually contains two elevation fields - * **UPPER** and * **LOWER**. However, if only one surface is required (as in a model of topography) then the other elevation field may be entered as - or dummy; SURFIP will then set the top of the model as the * **UPPER** field, or the bottom of the model as the * **LOWER** field.

### Splitting into sub-cells in X and Y

Interpolation is carried out initially at the centres (in the X-Y plane) of each cell. In a second pass, for any pair of adjacent columns of cells where the maximum vertical distance between corresponding (upper or lower) surface points exceeds a defined threshold value, the sub-cells required at the positions of the upper and lower surfaces are sub-divided further by vertical divisions to give, in plan view, 2, 4, or 8 sub-cells in either X or Y independently, the number increasing with increasing estimated slope in X or Y. Thus, where the slope is high, a cell may be split into as many as 8*8 sub-cells. This method ensures that the surface is accurately defined where it changes rapidly, without imposing this level of detail (and a large storage overhead) on the entire model. The maximum split possible is controlled by optional parameter @SPLITS; this may have the following values:-

* = 0 No splitting possible (cells only)

* = 1 A maximum of 2 possible in X or Y

* = 2 A maximum of 4 possible in X or Y

* = 3 A maximum of 8 possible in X or Y. This is the default.

The default value for the threshold is 0.5 of the average seam thickness as obtained during the first pass. This may be overridden by specifying a value for the optional @**MAXSTEP** parameter, which can be set to the maximum thickness difference (the threshold) permitted between adjacent cell centre surface values before splitting takes place.

Note that setting @**MAXSTEP** =0 has a special meaning. All cells will be divided into the maximum number of sub-cells allowed by the @**SPLITS** parameter (default 8*8) in the output model. If @**MAXSTEP** >0, then whole cells lying completely between, above or below the surfaces will be output as complete cells, even if cells lying on the surface are sub-divided in X and/or Y.

Although one of the two fields * **UPPER** and * **LOWER** may be implicit, it is not permissible to have both fields so. If a seam model is required with both surfaces horizontal then the sample data file should contain a single record with the appropriate * **UPPER** and * **LOWER** values, @**RADIUS** should be set sufficiently large, and @**MINNOP** should be set to l.

Interpolation proceeds by column. A progress message is produced at the start and end of interpolation of each column, in each interpolation pass.

If the &IN file is sorted on X, then **SURFIP** recognises this and searches the data faster. The saving in execution time on interpolating a surface from a large amount of data can be substantial.

If multiple perimeters have been used in interpolation, the following message is displayed:

>>> Interpolation has been with more than one
>>> perimeters, and therefore the model may require
>>> sorting on IJK before proceeding

### Surface identification in the output file

The output surface model file contains a numeric field (the * **LABEL** field) used to encode seam name or rock type. The user supplies values for this field for sub-cells lying above, within, or below the seam (the @**ABOVE** , @**WITHIN** or @**BELOW** parameters). Any one or more (but not all three) of these codes may be 'absent data' (-) if desired. For conventional seam modeling, to allow combination of multiple seams, it will usually be appropriate to code 'above' as the number chosen for a waste rock type, 'within' as the particular seam code, and 'below' as missing data; the process **OVRMOD** may then be used to combine seam models.

For surface topography models, conventionally only the @WITHIN parameter is set, and the @**ABOVE** and @BELOW parameters set to absent data (-). Often the code 99 is used for air (@**WITHIN** =99), but the user may choose his or her own value.

### The Trend File

The **SURFIP** process also has the capability to use a trend surface file as input (optional file &**TREND**). This file, consisting of a **DD** only, is output by process **TREND** ; it contains the coefficients of the trend surface computed from the same data points to be input to **SURFIP** on symbolic file &**IN**.

It is used in the following way: first, all the data points have the value of the trend surface at that point subtracted from them. Then these residuals are interpolated, giving estimates of trend surface residuals at cell and sub-cell centers; finally the value of the trend surface at each cell or sub-cell centre is added back to the residuals, to give elevations. The resulting surface has the general curvature of the trend surface, but passes through the original points.

Thus it is possible to obtain interpolated values which are higher than the highest data point (e.g. an anticline), or lower than the lowest data point (e.g. a syncline), and outside the region of data extrapolation will give a surface with the same shape as the trend surface, not a flat plane as would otherwise be the case.

### The Perimeter File

SURFIP may optionally use a perimeter file to control interpolation. This file should be in standard perimeter file format (containing numeric fields **XP** , **YP** , **ZP** , **PTN** and **PVALUE**) and can contain any number of perimeters, unclosed, and entered clockwise.

A separate interpolation will be carried out for each perimeter in the file; for the upper surface (hanging wall) only data points lying within the perimeter will be selected, and only cells and sub-cells lying inside the perimeter will be generated. However, thickness values will be taken from data points not only within the perimeter, but also from points lying within an @**RADIUS** from the (sub-)cell centre outside the perimeter. This is to allow continuity in thickness interpolation across a fault boundary.

This feature permits sub-crop or interpolation limits to be entered as a perimeter; or faults may be specified as a set of adjacent perimeters, so that each fault zone can be separately interpolated. The trend surface (&**TREND** file) may be used at the same time as the &**PERIMIN** file.

Cells will always be divided into sub-cells along the boundary of each perimeter, to the maximum number specified by the @**SPLITS** parameter; only those sub-cells whose centers lie within the perimeter will be interpolated and output.

Perimeters must abut each other accurately to ensure that there are not missing parts of the surface at the boundary; and perimeters must not overlap, or the same sub-cells will appear twice in the output file.

The output surface model file will require sorting on field IJK after generation, if more than one perimeter was in the input file. This is because the cells and sub-cells are generated in separate passes for each perimeter, and added to the end of the file. However, within each perimeter, cells and sub-cells will be generated in IJK order.

If a perimeter file was used, the output surface model will contain an extra field **PVALUE** , giving the perimeter identifier **PVALUE** within which the part of the surfaces lie. This may be used for surface classification or selection of part of the model later.

### Input Files:

* **proto** (Block Model Prototype):
  Prototype model. Must contain at least the fields **XC, YC, ZC, XINC, YINC, ZINC,

## XMORIG, YMORIG, ZMORIG, NX, NY, NZ, IJK.**

  Required=Yes

* **in** (Undefined):
  Input intersection data. Must contain the fields **X , Y , UPPER , LOWER**.
  Required=Yes

* **trend** (Undefined):
  Trend coefficients file (as produced from process TREND) defining a surface to be
  subtracted from the data before interpolation and added back to the interpolated values
  for each cell or sub-cell. The field names are **C0, CX, CY, CXY, CX2, CY2, CX2Y, CY2X**
  etc.
  Required=No

* **perimin** (String):
  Input perimeter file defining fault boundaries, or surface limits. One pass through the
  interpolation process is made for each perimeter on file, using only data lying within
  the perimeter, and generating only (sub-)cells lying within this perimeter. At the
  boundary, cells will be split into sub-cells, controlled by the **SPLITS** parameter.
  Required=No

### Output Files:

* **model** (Block Model):
  Output interpolated seam model.
  Required=Yes

### Fields:

* **x** (Numeric : IN):
  Name of intersection X field.
  Default=Undefined
  Required=Yes

* **y** (Numeric : IN):
  Name of intersection Y field.
  Default=Undefined
  Required=Yes

* **upper** (Numeric : Undefined):
  Name of intersection roof elevation field. Enter - or dummy if only **LOWER** supplied.
  Default=Undefined
  Required=Yes

* **lower** (Numeric : Undefined):
  Name of intersection floor elevation field. Enter - or dummy if only **UPPER**
  supplied.
  Default=Undefined
  Required=Yes

* **label** (Numeric : MODEL):
  Name of numeric field to be generated with values corresponding to **ABOVE , WITHIN ,

## BELOW**.

  Default=Undefined
  Required=Yes

### Parameters:

* **radius**:
  Search radius.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=Yes

* **above**:
  Value of **LABEL** above seam.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **within**:
  Value of **LABEL** within seam.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **below**:
  Value of **LABEL** below seam.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **power**:
  Weighting power (2).
  Range=Undefined
  Values=Undefined
  Default=2
  Required=No

* **minnop**:
  Minimum number of samples (5).
  Range=Undefined
  Values=Undefined
  Default=5
  Required=No

* **maxstep**:
  Max. elevation difference before sub-cells interpolated in horizontal plane [0.5 seam
  thickness]. If **MAXSTEP** is exactly 0, then splitting will take place over the entire
  model generated, as controlled by SPLITS.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **splits**:
  Controls splitting of sub-cells. The maximum number of sub-cells will be 2** **SPLITS**
  in X and Y. Default =3 [i.e. 2**3] = a max of 8*8 subcells in the XY plane (3).
  Range=1,3
  Values=1,2,3
  Default=3
  Required=No

* **print**:

* **Options** (1: summary of parameters and average seam thickness displayed (0).):
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('surfip')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute surfip
print("Running surfip...")
dm_cmd.surfip(
    proto_i='t_mod1',  # required
    in_i='t_assays',  # required
    perimin_i='optional',  # required
    model_o='t_surfip_out',  # required
    x_f='X',  # required
    y_f='Y',  # required
    upper_f='optional',  # required
    lower_f='optional',  # required
    label_f='optional',  # required
    radius_p='required_val',  # required
    # trend_i='optional',  # optional
    # above_p='optional',  # optional
    # within_p='optional',  # optional
    # below_p='optional',  # optional
    # power_p=2,  # optional
    # minnop_p=5,  # optional
    # maxstep_p='optional',  # optional
    # splits_p=3,  # optional
    # print_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("surfip execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_surfip_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")